# Safe Autonomy Tutorial 2: Motion Planning and Hamilton-Jacobi Reachability

## E-level, trajectory optimization

In [ ]:
# Dependencies (Colab has numpy, matplotlib and scipy preinstalled)
!pip install -q casadi shapely

In [ ]:
import math
from copy import copy, deepcopy

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, Ellipse, Circle
from matplotlib.collections import PatchCollection
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from shapely.geometry import Polygon as ShapelyPolygon, Point

# Map and robot parameters.
obstacles = [np.array(o) for o in [   # polygon vertices [x, y] in m, one list per obstacle
    [[2.633, 1.267], [3.033, 1.867], [3.833, 0.867]],
    [[1.575, 3.862], [1.175, 4.462], [2.375, 4.862], [2.375, 4.062]],
    [[3.75, 3.55], [4.95, 4.35], [5.55, 3.75], [4.75, 3.35]],
    [[7.35, 3.25], [7.95, 4.05], [7.95, 2.45]],
    [[6.238, 0.662], [5.638, 1.262], [6.038, 1.862], [6.838, 1.462]],
    [[4.217, 6.3], [5.017, 7.1], [6.017, 6.1]],
]]
x0 = np.array([1.25, 1.25, 0.785398])   # initial state [x, y, theta], in m and rad
xG = np.array([7.9, 6.6, 0.0])          # goal state [x, y, theta], in m and rad
MAP_W = 8.75          # map width, in m
MAP_H = 7.5           # map height, in m
ROBOT_RADIUS = 0.2    # the robot is overapproximated by a disk of this radius, in m
V_MIN = 0.0           # speed bounds, in m/s
V_MAX = 1.0
W_MAX = 1.0           # turn-rate bound, in rad/s
D_MAX = 0.1           # disturbance bound per axis, in m/s
N = 130               # number of trajectory points
DT = 0.1              # time step, in s
print(f"map {MAP_W} x {MAP_H} m, {len(obstacles)} obstacles, robot radius {ROBOT_RADIUS} m, horizon {N * DT:.0f} s")

## E0. The planning problem

We want a trajectory from an initial state to a goal state that respects the robot dynamics, avoid collision with the obstacles, and is optimal with respect to a performance cost. In discrete time with $N$ steps, we can formulate the following optimization problem:

$$
\begin{aligned}
	\min_{\mathbf{u}_{0:N-2},\, \mathbf{x}_{0:N-1}} \quad &\sum_{k=0}^{N-1} J(\mathbf{x}_k, \mathbf{u}_k)\\
	\textrm{s.t.} \quad  &\mathbf{x}_{k+1} = \mathbf{f}\left(\mathbf{x}_k, \mathbf{u}_k\right) &&k = 0, \dots, N-2\\
  &\mathbf{x}_{k}  \notin \mathcal{X}_{\mathrm{obs}} &&k = 0, \dots, N-1\\
  & \mathbf{u}_k \in \mathcal{U} &&k = 0, \dots, N-2\\
	& \mathbf{x}_{N-1} \in \mathcal{X}_g, \quad \mathbf{x}_0 = \mathbf{x}(t_0),
\end{aligned}
$$

where $\mathbf{x}_k \in \mathbb{R}^3$ is the robot state at step $k$, $\mathbf{u}_k \in \mathbb{R}^2$ the input, $\mathbf{f}: \mathbb{R}^3 \times \mathbb{R}^2 \to \mathbb{R}^3$ the discrete-time dynamics, which we assume to be a smooth function, $J$ the stage cost, $\mathcal{X}_{\mathrm{obs}}$ the set of states in collision, $\mathcal{U}$ the set of admissible inputs and $\mathcal{X}_g$ the goal set.

In this tutorial, we consider a wheeled mobile robot with unicycle kinematics. Its state collects the position and the heading, $\mathbf{x} = [x, y, \theta]^T$, and its input the speed and the turn rate, $\mathbf{u} = [v, \omega]^T$. We measure the heading from the positive $x$ axis, so $\theta = 0$ means the robot moves in the positive $x$ direction and a positive $\omega$ turns it counterclockwise. With a forward Euler discretization at time step $\Delta t$ the dynamics is

$$
\begin{bmatrix} x_{k+1}\\ y_{k+1}\\ \theta_{k+1} \end{bmatrix} =
\begin{bmatrix} x_k + v_k \,\Delta t \cos\theta_k\\ y_k + v_k \,\Delta t \sin\theta_k\\ \theta_k + \omega_k \,\Delta t \end{bmatrix},
\qquad \mathcal{U} = \left\{[v, \omega]^T \mid 0 \le v \le v_{\max},\ |\omega| \le \omega_{\max}\right\}.
$$

We allow zero forward speed and bound the turn rate, so the robot can stop or rotate in place. The figure below illustrates the model, with the heading $\theta$ measured from the positive $x$ axis, the speed $v$ along the heading, the turn rate $\omega$ about the center, and the footprint of radius $r$ that we use for collision checking.

<figure>
  
</figure>


In [ ]:
def plot_heading(ax, traj, every=10, color="k", scale=0.6):
    # Arrows along the heading of the robot every `every` steps of a (3, N) trajectory.
    th = traj[2, ::every]
    ax.quiver(traj[0, ::every], traj[1, ::every], np.cos(th), np.sin(th), color=color, angles="xy", scale_units="xy", scale=1 / scale, width=0.006)

The robot operates on a rectangular map with convex polygonal obstacles. The plot below shows the map and the robot at its start position.

In [ ]:
def map_axes(ax=None, title=None, footprint=True):
    # Draws the obstacles, start and goal. Returns the axes for further plotting.
    if ax is None:
        _, ax = plt.subplots(figsize=(6, 6 * MAP_H / MAP_W))
    ax.add_collection(PatchCollection([Polygon(o) for o in obstacles], color=[1, 0, 0], alpha=0.6))
    if footprint:
        ax.add_patch(Circle(x0[:2], ROBOT_RADIUS, color="b", alpha=0.3))
    ax.quiver(x0[0], x0[1], np.cos(x0[2]), np.sin(x0[2]), color="k", angles="xy", scale_units="xy", scale=1 / 0.6, width=0.006)
    ax.plot(x0[0], x0[1], "bo", label="start")
    ax.plot(xG[0], xG[1], "g*", markersize=12, label="goal")
    ax.set_xlim(0, MAP_W); ax.set_ylim(0, MAP_H); ax.set_aspect("equal")
    ax.set_xlabel("x in [m]"); ax.set_ylabel("y in [m]")
    if title: ax.set_title(title)
    return ax

map_axes(title="The map");
plt.show()

## E1. From the optimal control problem to a smooth nonlinear program

If the cost and all constraints in the optimization problem above are smooth functions of the decision variables, the problem is a smooth nonlinear program (NLP). An off-the-shelf gradient-based solver such as IPOPT [1] can then be used to solve it.

The dynamics and the input bounds are already smooth. However, the obstacle constraint $\mathbf{x}_k \notin \mathcal{X}_{\mathrm{obs}}$ is not. We therefore over-approximate each polygonal obstacle $i$ by its **minimum volume enclosing ellipse** (MVEE),

$$
\mathcal{E}_i = \left\{ \mathbf{p} \in \mathbb{R}^2 \mid (\mathbf{p} - \mathbf{c}_i)^T \mathbf{A}_i (\mathbf{p} - \mathbf{c}_i) \le 1 \right\},
$$

with center $\mathbf{c}_i \in \mathbb{R}^2$ and positive definite shape matrix $\mathbf{A}_i \in \mathbb{R}^{2 \times 2}$. 

The obstacle constraint then becomes

$$
(\mathbf{p}_k - \mathbf{c}_i)^T \mathbf{A}_i (\mathbf{p}_k - \mathbf{c}_i) \ge 1 \qquad \text{for every obstacle } i, \quad k = 0, \dots, N-1,
$$

which is smooth in the position $\mathbf{p}_k = [x_k, y_k]^T$. Because $\mathcal{E}_i$ contains polygon $i$, every position that satisfies this constraint is collision-free.

An ellipse contains a polygon exactly when it contains the polygon's vertices, so we compute the MVEE of the vertices with the Khachiyan algorithm in the function below. See Chapter 8.4.1 of [2] for the underlying problem. To plot an ellipse we use its semi-axes: the eigenvectors of $\mathbf{A}_i$ give the directions of the semi-axes, and their lengths are $1/\sqrt{\lambda}$ for the corresponding eigenvalues $\lambda$.

In [ ]:
def mvee(points, tol=1e-4):
    # Minimum volume enclosing ellipsoid in center form (p - c)^T A (p - c) <= 1.
    n, d = points.shape
    Q = np.column_stack((points, np.ones(n))).T
    u = np.ones(n) / n
    err = tol + 1.0
    while err > tol:
        X = Q @ np.diag(u) @ Q.T
        M = np.diag(Q.T @ np.linalg.inv(X) @ Q)
        j = np.argmax(M)
        step = (M[j] - d - 1.0) / ((d + 1) * (M[j] - 1.0))
        new_u = (1 - step) * u
        new_u[j] += step
        err = np.linalg.norm(new_u - u)
        u = new_u
    c = u @ points
    A = np.linalg.inv(points.T @ np.diag(u) @ points - np.outer(c, c)) / d
    return A, c

def ellipse_patch(A, c, color):
    lam, R = np.linalg.eigh(A)
    width, height = 2 / np.sqrt(lam)
    angle = math.degrees(math.atan2(R[1, 0], R[0, 0]))
    return Ellipse(c, width, height, angle=angle, facecolor="none", edgecolor=color)

obstacle_ellipses = [mvee(o) for o in obstacles]   # list of (A, c)

ax = map_axes(title="Minimum volume enclosing ellipses")
for A, c in obstacle_ellipses:
    ax.add_artist(ellipse_patch(A, c, [1, 0, 0]))
plt.show()

So far we have treated the robot as a point, but its footprint is a disk of radius $r$. The robot is therefore collision-free if its center keeps a distance of at least $r$ from every ellipse.  We obtain this by enlarging the ellipses once, by replacing each ellipse $\mathcal{E}_i$ with a larger ellipse $\tilde{\mathcal{E}}_i$, with shape matrix $\tilde{\mathbf{A}}_i$ and center $\tilde{\mathbf{c}}_i$, that contains every point within distance $r$ of $\mathcal{E}_i$:

$$
\mathbf{p} \notin \tilde{\mathcal{E}}_i \implies \mathrm{dist}(\mathbf{p}, \mathcal{E}_i) \ge r.
$$


For the rest of the notebook we can treat the robot as a point.

**Exercise E1.** Implement `enlarge(A, c, radius)`, which returns the shape matrix $\tilde{\mathbf{A}}$ and center $\tilde{\mathbf{c}}$ of an enlarged ellipse to account for the robot radius.

*Hint.* Use an eigendecomposition of $\mathbf{A}$ to write the ellipse in its normal form $(x/a)^2 + (y/b)^2 = 1$ with semi-axes $a \ge b$.

In [ ]:
def enlarge(A, c, radius):
    # Ellipse that contains all points within `radius` of the ellipse (A, c).
    # BEGIN SOLUTION
    return None, None   # shape matrix and center of the enlarged ellipse
    # END SOLUTION

enlarged_ellipses = [enlarge(A, c, ROBOT_RADIUS) for A, c in obstacle_ellipses]

# Check: points on the boundary of the enlarged ellipse are at least ROBOT_RADIUS away from the polygon.
def boundary_points(A, c, n=200):
    lam, R = np.linalg.eigh(A)
    t = np.linspace(0, 2 * np.pi, n)
    return c + (R @ (np.stack([np.cos(t), np.sin(t)]) / np.sqrt(lam)[:, None])).T

for (A, c), poly in zip(enlarged_ellipses, obstacles):
    d = [ShapelyPolygon(poly).distance(Point(p)) for p in boundary_points(A, c)]
    assert min(d) >= ROBOT_RADIUS - 1e-6, "an enlarged ellipse comes closer than the robot radius to its polygon"

ax = map_axes(title="Enlarged ellipses (blue) around the obstacle ellipses (red), robot as a point", footprint=False)
for (A, c) in obstacle_ellipses:
    ax.add_artist(ellipse_patch(A, c, [1, 0, 0]))
for (A, c) in enlarged_ellipses:
    ax.add_artist(ellipse_patch(A, c, [0, 0, 1]))
plt.show()

> **Reflection:** Could replacing polygons with enlarged ellipses rule out a trajectory that is safe?

## E2. Trajectory optimization with static obstacles

We now have the smooth obstacle constraint with the enlarged ellipses $\tilde{\mathcal{E}}_i$, for every obstacle $i$ and every step $k$. As a cost we use the quadratic stage cost

$$
J(\mathbf{x}_k, \mathbf{u}_k) = (\mathbf{x}_k - \mathbf{x}_G)^T \mathbf{Q} (\mathbf{x}_k - \mathbf{x}_G) + \mathbf{u}_k^T \mathbf{R} \mathbf{u}_k,
\qquad \mathbf{Q} = \mathbf{I}_{3\times 3},\quad \mathbf{R} = \mathrm{diag}(2, 2),
$$

and require the final position to coincide with the goal position $\mathbf{x}_G$. The heading at the goal is not constrained.

We write the NLP with `casadi`, a modeling framework in which we declare the decision variables, express the cost and the constraints symbolically, and then call the solver. Its `Opti` interface lets us declare variables, cost and constraints one by one and choose the solver at the end (here we use IPOPT).

**Exercise E2.** Complete `solve_trajopt`, which formulates and solves the problem with `casadi.Opti`, and report the time the solver needs.

*Hint.* Use `casadi.sin` and `casadi.cos` in the dynamics. When calling the function, use the input bounds `V_MIN`, `V_MAX` and `W_MAX`, and the enlarged ellipses in the obstacle constraint. A linear interpolation between start and goal is a reasonable initial guess for the solver.


In [ ]:
import casadi

def solve_trajopt(x0, xG, ellipses_per_step, v_min, v_max, w_max, N, dt, Q=np.eye(3), R=2 * np.eye(2)):
    # ellipses_per_step[k] is the list of (A, c) ellipses the robot keeps outside of at step k.
    opti = casadi.Opti()
    X = opti.variable(3, N)
    U = opti.variable(2, N - 1)
    # BEGIN SOLUTION
    return None, None   # state trajectory (3, N) and inputs (2, N-1)
    # END SOLUTION

import time
t0 = time.time()
traj, inputs = solve_trajopt(x0, xG, [enlarged_ellipses] * N, V_MIN, V_MAX, W_MAX, N, DT)
print(f"solved in {time.time() - t0:.2f} s")

def clearance(traj, polygons_per_step):
    # Smallest distance between the robot center and any polygon along the trajectory.
    return min(ShapelyPolygon(poly).distance(Point(traj[0, k], traj[1, k]))
               for k in range(traj.shape[1]) for poly in polygons_per_step[k])

assert traj.shape == (3, N) and inputs.shape == (2, N - 1)
assert np.all(inputs[0] >= V_MIN - 1e-6) and np.all(inputs[0] <= V_MAX + 1e-6) and np.all(np.abs(inputs[1]) <= W_MAX + 1e-6)
assert clearance(traj, [obstacles] * N) >= ROBOT_RADIUS - 1e-6, "the trajectory comes closer than the robot radius to an obstacle"
print(f"minimum clearance {clearance(traj, [obstacles] * N):.3f} m, final position error {np.linalg.norm(traj[0:2, -1] - xG[0:2]):.2e} m")

We plot the trajectory of the robot center together with both sets of ellipses, with arrows marking the heading, and then animate the robot along it.

In [ ]:
ax = map_axes(title="Optimized trajectory", footprint=False)
for (A, c) in obstacle_ellipses:
    ax.add_artist(ellipse_patch(A, c, [1, 0, 0]))
for (A, c) in enlarged_ellipses:
    ax.add_artist(ellipse_patch(A, c, [0, 0, 1]))
ax.plot(traj[0], traj[1], "b-")
plot_heading(ax, traj, every=8, color="k")
plt.show()

def animate(traj, polygons_per_step, ellipses_per_step, title=""):
    # Animation of the robot (its center and a heading arrow), the obstacles and their enlarged ellipses, which move if they do.
    fig, ax = plt.subplots(figsize=(6, 6 * MAP_H / MAP_W))
    patches = [Polygon(p, color=[1, 0, 0], alpha=0.6) for p in polygons_per_step[0]]
    for p in patches:
        ax.add_patch(p)
    ellipses = [ellipse_patch(A, c, [0, 0, 1]) for A, c in ellipses_per_step[0]]
    for e in ellipses:
        ax.add_patch(e)
    ax.plot(xG[0], xG[1], "g*", markersize=12)
    path, = ax.plot([], [], "g-", lw=1)
    body, = ax.plot([], [], "bo")
    heading = ax.quiver([0.], [0.], [0.], [1.], color="k", angles="xy", scale_units="xy", scale=1 / 0.6, width=0.006)
    ax.set_xlim(0, MAP_W); ax.set_ylim(0, MAP_H); ax.set_aspect("equal"); ax.set_title(title)

    def frame(i):
        x, y, th = traj[:, i]
        body.set_data([x], [y])
        heading.set_offsets([[x, y]]); heading.set_UVC([math.cos(th)], [math.sin(th)])
        path.set_data(traj[0, :i + 1], traj[1, :i + 1])
        for p, verts in zip(patches, polygons_per_step[i]):
            p.set_xy(verts)
        for e, (A, c) in zip(ellipses, ellipses_per_step[i]):
            e.set_center(c)
        return [body, heading, path, *patches, *ellipses]

    anim = FuncAnimation(fig, frame, frames=traj.shape[1], interval=50, blit=True)
    plt.close(fig)
    return HTML(anim.to_jshtml())

animate(traj, [obstacles] * N, [enlarged_ellipses] * N, "Static obstacles")

We save this open-loop plan as the nominal plan, which the C/A notebook loads to compare against the controllers designed there.

In [ ]:
# Save the nominal plan next to this notebook, for the C/A notebook: rows 0-2 states, rows 3-4 inputs (the last input is repeated).
np.save("plan_nominal.npy", np.vstack([traj, np.hstack([inputs, inputs[:, -1:]])]))

## E3. Trajectory optimization with moving obstacles

Obstacles in a real environment move. In this tutorial we assume that we know their motion, specifically that every obstacle moves with a constant velocity, given in the cell below. The polygon of an obstacle at step $k$ is then its initial polygon translated by $k\,\Delta t$ times its velocity, and the obstacle constraint of step $k$ uses the ellipse of step $k$.

**Exercise E3a.** Build the lists of obstacle polygons and of enlarged ellipses for every step of the horizon. 

*Hint.* Translating an ellipse moves its center and leaves its shape matrix unchanged.


In [ ]:
obstacle_velocities = [np.array(v) for v in [[0.15, 0.0], [0.2089, 0.0836], [0.0557, -0.1393], [-0.5367, 0.2683], [-0.0448, 0.4478], [0.0735, 0.0147]]]   # translation velocity [vx, vy] of each polygon, in m/s

# BEGIN SOLUTION
timed_obstacles = []   # entry k: the obstacle polygons at step k
timed_ellipses = []    # entry k: the enlarged ellipses (A, c) at step k
# END SOLUTION

assert len(timed_obstacles) == N and len(timed_ellipses) == N
assert np.allclose(timed_obstacles[0][0], obstacles[0]) and np.allclose(timed_ellipses[-1][1][1], enlarged_ellipses[1][1] + (N - 1) * DT * obstacle_velocities[1])
animate(np.full((3, N), np.nan), timed_obstacles, timed_ellipses, "Moving obstacles")

**Exercise E3b.** Solve the planning problem with the moving obstacles by calling `solve_trajopt` with the ellipses per step from exercise E3a. Animate the result.

In [ ]:
# BEGIN SOLUTION
traj_moving, inputs_moving = None, None   # state trajectory (3, N) and inputs (2, N-1)
# END SOLUTION

assert traj_moving.shape == (3, N)
assert clearance(traj_moving, timed_obstacles) >= ROBOT_RADIUS - 1e-6
print(f"minimum clearance to the moving obstacles {clearance(traj_moving, timed_obstacles):.3f} m")
animate(traj_moving, timed_obstacles, timed_ellipses, "Moving obstacles")

> **Reflection:** Suppose the robot follows the same geometric path at a different speed. Could a path that was safe become unsafe? How this differs between static and moving obstacles?

## E4. Adding a disturbance

So far we computed the plan once before execution (open loop), for a model that we assumed to be exact. We return to the **static obstacles** of E2 and add a disturbance $\mathbf{d} = [d_x, d_y]^T$ to the velocity of the position,

$$
\dot x = v \cos\theta + d_x, \qquad \dot y = v \sin\theta + d_y, \qquad |d_x|, |d_y| \le d_{\max}.
$$

The disturbance $\mathbf{d}$ lies in the box $\mathcal{D} = \{[d_x, d_y]^T \mid |d_x| \le d_{\max},\ |d_y| \le d_{\max}\}$.
In the cell below, we apply the inputs of the nominal plan from E2 with a constant disturbance from each corner of the disturbance box and report which runs collide.

In [ ]:
def rollout_open_loop(x_start, inputs, d, dt):
    # Forward Euler simulation of the unicycle with a constant disturbance vector.
    X = np.zeros((3, inputs.shape[1] + 1))
    X[:, 0] = x_start
    for k in range(inputs.shape[1]):
        v, w = inputs[:, k]
        th = X[2, k]
        X[:, k + 1] = X[:, k] + dt * np.array([v * math.cos(th) + d[0], v * math.sin(th) + d[1], w])
    return X

ax = map_axes(title=f"Open-loop plan under a steady disturbance of {D_MAX} m/s per axis")
ax.plot(traj[0], traj[1], "b-", label="plan")
for sx, sy in [(1, 1), (1, -1), (-1, 1), (-1, -1)]:
    d = D_MAX * np.array([sx, sy])
    X = rollout_open_loop(x0, inputs, d, DT)
    collides = clearance(X, [obstacles] * N) < ROBOT_RADIUS
    ax.plot(X[0], X[1], "r-" if collides else "g-", alpha=0.8, label=f"disturbance ({sx:+d}, {sy:+d}): {'collision' if collides else 'safe'}")
ax.legend(fontsize=8)
plt.show()

We see that at least one direction of the disturbance drives the robot into an obstacle.

> **Reflection:** Why the collision can happen even though the plan keeps a distance of at least the robot radius to every obstacle?

## References

[1] Wächter, A. and Biegler, L. T. (2006). On the implementation of an interior-point filter line-search algorithm for large-scale nonlinear programming. *Mathematical Programming*, 106(1), 25–57.

[2] Boyd, S. and Vandenberghe, L. (2004). *Convex Optimization*. Cambridge University Press.